# 05 — Revenue Analysis & Forecasting
Monthly/quarterly trends, growth decomposition, 3–6 month revenue forecast.

**Input:** `data/featured/superstore_orders.csv`

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

orders = pd.read_csv('data/featured/superstore_orders.csv')
orders['order_date'] = pd.to_datetime(orders['order_date'])
monthly = orders.groupby(orders['order_date'].dt.to_period('M')).agg(
    Sales=('order_total_sales', 'sum'),
    Profit=('order_total_profit', 'sum'),
    Orders=('Order ID', 'nunique')
).reset_index()
monthly['order_date'] = monthly['order_date'].dt.to_timestamp()
monthly

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly['order_date'], monthly['Sales'], label='Sales')
ax2 = ax.twinx()
ax2.plot(monthly['order_date'], monthly['Profit'], color='orange', label='Profit')
ax.legend(loc='upper left'); ax2.legend(loc='upper right')
ax.set_ylabel('Sales ($)'); ax2.set_ylabel('Profit ($)')
plt.title('Monthly Revenue & Profit Trend')
plt.tight_layout()
plt.show()

In [ ]:
from src.forecasting import prepare_monthly_series, fit_trend, forecast_months

monthly_df = prepare_monthly_series(orders)
model, r2, pred = fit_trend(monthly_df, 'Sales')
print(f'Linear trend R² (Sales): {r2:.4f}')
last_ord = monthly_df['month_ordinal'].max()
fc = forecast_months(model, last_ord, n_months=6)
print('Next 6 months Sales forecast:', fc.round(0))